# FLUX Q8 双开 — 网络下载路线
不挂数据集：权重全部从 HF 现下（掐表），本地组装 diffusers 布局，编码后双开出图。

In [ ]:
# ===== 参数区 =====
PROMPTS = [
    "G.E.M. Deng Ziqi, famous Chinese pop singer, singing passionately on a beach at golden sunset, holding a microphone, ocean waves and palm trees in the background, warm golden hour light, photorealistic portrait, 85mm lens, ultra detailed",
    "G.E.M. Deng Ziqi, famous Chinese pop singer, performing live on a concert stage, holding a microphone, dramatic purple and blue stage lights, cheering crowd silhouettes, photorealistic, ultra detailed",
]
SEEDS = [42, 43]
STEPS, GUIDANCE, W, H = 28, 3.5, 1024, 1024
STAGGER_S = 60

In [ ]:
# ===== 下载全部权重（逐项掐表）=====
import subprocess, sys, os, time, glob, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf==0.19.0"], check=True)
import gguf  # 先于 diffusers
import torch
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = tok
from huggingface_hub import snapshot_download, hf_hub_download
BASE = "/kaggle/working/flux_local"; os.makedirs(BASE, exist_ok=True)
T_INGRESS = time.time()
def timed(label, fn):
    t0 = time.time(); r = fn(); print(f"[计时] {label}: {time.time()-t0:.1f}s"); return r
timed("小件 CLIP/VAE/tokenizer/配置", lambda: snapshot_download(
    "black-forest-labs/FLUX.1-dev", local_dir=BASE, token=tok,
    allow_patterns=["text_encoder/**", "vae/**", "tokenizer/**", "tokenizer_2/**",
                    "scheduler/**", "model_index.json", "transformer/config.json",
                    "text_encoder_2/config.json", "text_encoder_2/generation_config.json"]))
for f in glob.glob(BASE + "/text_encoder_2/*.safetensors"): os.remove(f)
p = timed("T5 fp8", lambda: hf_hub_download("comfyanonymous/flux_text_encoders", "t5xxl_fp8_e4m3fn.safetensors", local_dir="/kaggle/working/hf_t5"))
shutil.move(p, BASE + "/text_encoder_2/model.safetensors")  # move 而非 copy：T5 双份占盘会撑爆 19.5G 工作区（实测教训）
GGUF_PATH = timed("GGUF Q8_0", lambda: hf_hub_download("city96/FLUX.1-dev-gguf", "flux1-dev-Q8_0.gguf", local_dir=BASE))
DS = BASE
print(f"[计时] 全部下载总计: {time.time()-T_INGRESS:.1f}s")
print("DS:", DS); print(sorted(os.listdir(DS)))
print("GPUs:", torch.cuda.device_count())

In [ ]:
# ===== 编码（关梯度；编码器用完即卸）=====
from transformers import CLIPTextModel, CLIPTokenizer, T5EncoderModel, T5TokenizerFast
torch.set_grad_enabled(False)
tok1 = CLIPTokenizer.from_pretrained(DS, subfolder="tokenizer")
tok2 = T5TokenizerFast.from_pretrained(DS, subfolder="tokenizer_2")
t0 = time.time()
te1 = CLIPTextModel.from_pretrained(DS, subfolder="text_encoder", torch_dtype=torch.float16).to("cuda:0")
te2 = T5EncoderModel.from_pretrained(DS, subfolder="text_encoder_2", torch_dtype=torch.float16).to("cuda:0")
print(f"编码器加载: {time.time()-t0:.1f}s")
def encode(prompt):
    i1 = tok1(prompt, padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids.to("cuda:0")
    pooled = te1(i1).pooler_output
    i2 = tok2(prompt, padding="max_length", max_length=512, truncation=True, return_tensors="pt").input_ids.to("cuda:0")
    return te2(i2).last_hidden_state.cpu(), pooled.cpu()
os.makedirs("/kaggle/working/embeds", exist_ok=True)
for i, p in enumerate(PROMPTS):
    emb, pooled = encode(p)
    torch.save({"emb": emb, "pooled": pooled, "seed": SEEDS[i]}, f"/kaggle/working/embeds/emb_{i}.pt")
    print(i, emb.shape, pooled.shape)
del te1, te2; torch.cuda.empty_cache()
print(f"编码阶段完成，总计 {time.time()-t0:.1f}s（含加载）")

In [ ]:
# ===== worker 脚本 =====
worker = r"""
import sys, time, json
import gguf
import torch
from diffusers import FluxTransformer2DModel, FluxPipeline, AutoencoderKL, FlowMatchEulerDiscreteScheduler, GGUFQuantizationConfig
idx = int(sys.argv[1]); ds = sys.argv[2]; gguf_path = sys.argv[3]
t0 = time.time()
transformer = FluxTransformer2DModel.from_single_file(gguf_path, config=f"{ds}/transformer", quantization_config=GGUFQuantizationConfig(compute_dtype=torch.float16))
transformer = transformer.to("cuda:0")
vae = AutoencoderKL.from_pretrained(ds, subfolder="vae", torch_dtype=torch.float16).to("cuda:0")
sched = FlowMatchEulerDiscreteScheduler.from_pretrained(ds, subfolder="scheduler")
pipe = FluxPipeline(scheduler=sched, vae=vae, text_encoder=None, tokenizer=None, text_encoder_2=None, tokenizer_2=None, transformer=transformer)
t_load = time.time() - t0
d = torch.load(f"/kaggle/working/embeds/emb_{idx}.pt")
g = torch.Generator(device="cuda:0").manual_seed(int(d["seed"]))
t1 = time.time()
img = pipe(prompt_embeds=d["emb"].to("cuda:0", torch.float16), pooled_prompt_embeds=d["pooled"].to("cuda:0", torch.float16),
           width=1024, height=1024, num_inference_steps=28, guidance_scale=3.5, generator=g).images[0]
t_gen = time.time() - t1
img.save(f"/kaggle/working/out_{idx}.png")
print(json.dumps({"idx": idx, "load_s": round(t_load,1), "gen_plus_decode_s": round(t_gen,1), "peak_gb": round(torch.cuda.max_memory_allocated()/2**30,2)}))
"""
open("/kaggle/working/worker.py", "w").write(worker)
print("worker.py 已写入")

In [ ]:
# ===== 双开 =====
import subprocess
procs = []
t0 = time.time()
for k in range(len(PROMPTS)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
    procs.append(subprocess.Popen([sys.executable, "/kaggle/working/worker.py", str(k), DS, GGUF_PATH], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True))
    if k + 1 < len(PROMPTS): time.sleep(STAGGER_S)
for p in procs: print(p.communicate()[0])
wall = time.time() - t0
print(f"双开墙钟: {wall:.1f}s | 每图: {wall/len(PROMPTS):.1f}s")

In [ ]:
from IPython.display import Image, display
for i in range(len(PROMPTS)): display(Image(filename=f"/kaggle/working/out_{i}.png"))